# AnySport – train the hockey detector (free, Colab T4)

Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom. About 1–2 h for 60 epochs.

Everything used is free and Apache/MIT licensed: HockeyAI data (MIT), our own detector (Apache-2.0), PyTorch.

In [ ]:
!git clone --depth 1 https://github.com/VICCOM-devs/sports-cam.git anysport
%cd anysport
!pip -q install huggingface_hub onnx onnxscript pyyaml opencv-python-headless

## 1. Data
Downloads HockeyAI (~400 MB). To add your own labelled frames from the hub's labelling tool, upload the `labels` folder to Colab and add `--extra /content/labels`.

In [ ]:
!python ml/prepare_data.py --out ml/data

## 2. Train
`yolox-s` is the default (fast on the hub's CPU/GPU). Try `--model yolox-m` for accuracy or `--model rtdetr-r18` (needs `pip install transformers`) for the best puck detection.

In [ ]:
!python ml/train.py --data ml/data --model yolox-s --epochs 60 --batch 16 --img 960 --out ml/runs/yolox-s

## 3. Export for the hub

In [ ]:
!python ml/export.py --run ml/runs/yolox-s --out custom.onnx
from google.colab import files
files.download('custom.onnx'); files.download('custom.json')

## 4. Install
Copy `custom.onnx` and `custom.json` to the hub's models folder:

- Windows: `%LOCALAPPDATA%\VICCOM\AnySport\models\`
- macOS/Linux: `~/.anysport/models/`

Restart the hub. `http://localhost:8080/api/version` now shows `"model": "custom"`, and the viewer draws the puck.